Path setup

In [1]:
import sys
sys.path.append("..")

Imports

In [2]:
from src.similarity.vectorizer import TfidfSimilarityVectorizer
from src.similarity.similarity_scorer import SimilarityScorer
from src.similarity.batch_comparator import compare_students_to_master

Test the vectorizer in isolation

In [3]:
vectorizer = TfidfSimilarityVectorizer()

docs = [
    "the cell membrane controls what enters and exits the cell",
    "the cell membrane regulates substances entering and leaving the cell",
    "photosynthesis converts light energy into chemical energy in plants"
]

matrix = vectorizer.fit_transform(docs)
print("Matrix shape:", matrix.shape)
print("Vocabulary size:", len(vectorizer.vectorizer.vocabulary_))

2026-08-02 18:20:57 | INFO     | src.similarity.vectorizer | TF-IDF vectorizer initialized with ngram_range=(1, 2)


Matrix shape: (3, 42)
Vocabulary size: 42


Test the similarity scorer in isolation

In [4]:
scorer = SimilarityScorer()

# doc 0 and doc 1 are paraphrases (should score high)
score_similar = scorer.compute_score(matrix[0], matrix[1])
print("Similar docs score:", score_similar, "->", scorer.match_level(score_similar))

# doc 0 and doc 2 are unrelated (should score low)
score_different = scorer.compute_score(matrix[0], matrix[2])
print("Different docs score:", score_different, "->", scorer.match_level(score_different))

Similar docs score: 0.4645532032424639 -> Poor
Different docs score: 0.0 -> Poor


Sanity check

In [5]:
identical_docs = ["this is a test sentence", "this is a test sentence"]
identical_matrix = vectorizer.fit_transform(identical_docs)
score_identical = scorer.compute_score(identical_matrix[0], identical_matrix[1])
print("Identical text score:", score_identical)

Identical text score: 1.0000000000000004


Full pipeline test with real sample files

In [6]:
student_files = ["../data/samples/Word-Docx.docx", "../data/samples/Native-PDF.pdf"]
master_file = "../data/samples/Native-PDF.pdf"  # deliberately reuse one file as "student" too

df = compare_students_to_master(student_files, master_file)
df

2026-08-02 18:20:58 | INFO     | src.preprocessing.preprocessor | Loading spaCy model: en_core_web_sm
2026-08-02 18:21:13 | INFO     | src.similarity.vectorizer | TF-IDF vectorizer initialized with ngram_range=(1, 2)
2026-08-02 18:21:14 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf
2026-08-02 18:21:14 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples/Word-Docx.docx
2026-08-02 18:21:15 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf
2026-08-02 18:21:15 | INFO     | src.similarity.batch_comparator | ../data/samples/Word-Docx.docx: score=1.0000, level=Excellent
2026-08-02 18:21:15 | INFO     | src.similarity.batch_comparator | ../data/samples/Native-PDF.pdf: score=1.0000, level=Excellent


,filename,similarity_score,match_level
0,../data/samples/Word-Docx.docx,1.0,Excellent
1,../data/samples/Native-PDF.pdf,1.0,Excellent


Deliberate failure test

In [7]:
student_files_with_bad_file = [
    "../data/samples/Native-PDF.pdf",
    "../data/samples/does_not_exist.pdf"
]
df_with_error = compare_students_to_master(student_files_with_bad_file, master_file)
df_with_error

2026-08-02 18:21:15 | INFO     | src.preprocessing.preprocessor | Loading spaCy model: en_core_web_sm
2026-08-02 18:21:23 | INFO     | src.similarity.vectorizer | TF-IDF vectorizer initialized with ngram_range=(1, 2)
2026-08-02 18:21:23 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf
2026-08-02 18:21:24 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf
2026-08-02 18:21:24 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/does_not_exist.pdf
2026-08-02 18:21:24 | ERROR    | src.extraction.pdf_extractor | Failed to open or read PDF ../data/samples/does_not_exist.pdf: no such file: '../data/samples/does_not_exist.pdf'
Traceback (most recent call last):
  File "c:\Users\akers\Desktop\project\intelliscreen-nlp\notebooks\..\src\extraction\pdf_extractor.py", line 12, in extract_text_from_pdf
    with fitz.open(file_path) as

,filename,similarity_score,match_level
0,../data/samples/Native-PDF.pdf,1.0,Excellent


Inspect score distribution across all samples

In [8]:
import os

all_files = [os.path.join("../data/samples", f) for f in os.listdir("../data/samples")]
df_all = compare_students_to_master(all_files, master_file)
df_all.sort_values("similarity_score", ascending=False)

2026-08-02 18:21:25 | INFO     | src.preprocessing.preprocessor | Loading spaCy model: en_core_web_sm
2026-08-02 18:21:33 | INFO     | src.similarity.vectorizer | TF-IDF vectorizer initialized with ngram_range=(1, 2)
2026-08-02 18:21:33 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/Native-PDF.pdf
2026-08-02 18:21:34 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples\Native-PDF.pdf
2026-08-02 18:21:34 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples\PNG-Pic.png
2026-08-02 18:21:34 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples\PNG-Pic.png
2026-08-02 18:21:52 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples\Text-txt.txt
2026-08-02 18:21:52 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples\Word-Docx.docx
2026-08-02 18:21:52 |

,filename,similarity_score,match_level
0,../data/samples\Native-PDF.pdf,1.0,Excellent
1,../data/samples\PNG-Pic.png,1.0,Excellent
2,../data/samples\Text-txt.txt,1.0,Excellent
3,../data/samples\Word-Docx.docx,1.0,Excellent
